# GX_03
# Lenses

In this graded exercise you are going to walk through the simulation of a thick lens with the beam propagation method. You will start by looking at the thin lens approximation, then move up to a thick lens and compare the results. For this exercise, a full implementation of the beam propagation method has been provided. You will be responsible for creating the phase masks of the thin and thick lenses and integrating them with the simulation.

To get a sense of how the lenses perform, we will be simulating the imaging setup shown in class where the lens is placed at a distance of 2f away from our object, and the imaging plane is located at a distance of 2f past the lens.

Make sure you understand the BPM implementation that is provided, and when you are ready move to part 1.

In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

N = 512
dx = 2e-6
wavelength = 633e-9
k = 2*np.pi/wavelength

x = (np.arange(N) - N//2) * dx
y = (np.arange(N) - N//2) * dx

X, Y = np.meshgrid(x, y)


In [ ]:
# - No modification necessary -

def bpm_propagate(U0, dz, phase_slices=None, steps=None):
    """
    Unified BPM propagation.

    Parameters
    ----------
    U0 : 2D complex array
        Input field
    dz : float
        Propagation step size
    phase_slices : array-like or None
        Phase mask at each step (used for media / lens propagation)
    steps : int or None
        Number of propagation steps if no phase masks are used

    Returns
    -------
    cross_sections : array
        Intensity cross-sections during propagation
    U : 2D complex array
        Output field
    """

    # -------- error checking --------
    if phase_slices is None and steps is None:
        raise ValueError("Either phase_slices or steps must be provided.")

    if phase_slices is not None and steps is not None:
        raise ValueError("Provide either phase_slices OR steps, not both.")

    # determine number of iterations
    if phase_slices is not None:
        iterator = phase_slices
        n_steps = len(phase_slices)
    else:
        iterator = [None] * steps
        n_steps = steps

    # -------- precompute transfer function --------
    fx = np.fft.fftfreq(N, dx)
    fy = np.fft.fftfreq(N, dx)

    FX, FY = np.meshgrid(fx, fy)

    H = np.exp(-1j * dz * ((2*np.pi*FX)**2 + (2*np.pi*FY)**2) / (2*k))

    U = U0.copy()
    cross_sections = []

    # -------- propagation loop --------
    for phase in tqdm(iterator, total=n_steps):

        U = np.fft.ifft2(np.fft.fft2(U) * H)

        if phase is not None:
            U = U * np.exp(1j * phase)

        cross_sections.append(np.abs(U[N//2])**2)

    return np.array(cross_sections), U

In [ ]:
# - No modification necessary -

from skimage import data
from skimage.transform import resize

def load_input_field(scale=0.5):
    """
    Load a built-in skimage image and encode it as a
    phase-only optical field with uniform amplitude.

    Parameters
    ----------
    N : int
        Simulation grid 
    scale : float
        Fraction of the domain the image should occupy (0 < scale <= 1)

    Returns
    -------
    U0 : complex ndarray
        Complex optical field
    """

    # Load example image
    img = data.camera()

    # Determine size of scaled image
    scaled_size = int(N * scale)
    img_resized = resize(img, (scaled_size, scaled_size), anti_aliasing=True)
    img_resized = img_resized / img_resized.max()  # normalize to [0,1]
    U_img = img_resized

    # Embed in NxN grid
    U0 = np.zeros((N, N), dtype=complex)  # uniform amplitude outside image
    start = N//2 - scaled_size//2
    end = start + scaled_size
    U0[start:end, start:end] = U_img

    return U0

U0 = load_input_field()

plt.imshow(np.abs(U0))
plt.colorbar()
plt.show()

# Part 1
## Simulating the Thin Lens

Begin by implementing the routine thin_lens_phase, which should take a focal length f (in meters) and return the phase of a thin lens on the meshgrid X, Y. 

Recall that the phase induced by a lens as a function of x and y is given as:
$$
\phi(x, y) = - \frac{k}{2 f} \, (x^2 + y^2)
$$

After you have implemented this, you will need to incorporate the function thin_lens_phase into the propagate_thin_lens_system routine. This function simulates the system in the introduction, wherein a propagation of 2f is done, then the phase of the thin lens is applied, and then a second propagation of 2f is performed.

After you have integrated the thin lens into the propagation function, use the thin_lens_demo function to visualize your function and sanity check its output. Then, answer the following questions:

1) How do we approximate a lens as a single phase mask? Recall the derivation of the thin film approximation and class and use it to justify your answer.
2) Does the quality of the output match your expectations? What are the main factors that degrade the output image?


In [ ]:
def thin_lens_phase(f):
    """
    Generate thin lens phase profile.
    """
    raise NotImplementedError

In [ ]:
def propagate_thin_lens_system(U0, f, steps=200):
    dz = 2 * f / steps

    U = U0.copy()

    # -------- propagate to lens --------
    cross1, U = bpm_propagate(U, dz, steps=steps)

    # -------- apply thin lens --------
    # TODO
    
    # -------- propagate away --------
    cross2, U = bpm_propagate(U, dz, steps=steps)

    # combine cross sections
    cross_sections = np.vstack([cross1, cross2])

    return cross_sections, dz, U

In [ ]:
# - No modification necessary - 

def thin_lens_demo(U0, f):
    """
    Thin lens demo with input/output field and cross-section.
    """
    # Propagate through thin lens system
    cross, dz, U_final = propagate_thin_lens_system(U0, f)

    # Create subplots
    fig, axs = plt.subplots(1, 3, figsize=(18,5))

    # Input field (phase)
    im0 = axs[0].imshow(np.abs(U0), cmap="twilight")
    axs[0].set_title("Input Phase")
    plt.colorbar(im0, ax=axs[0])

    display_data = np.log10(cross + 1e-12)

    im2 = axs[1].imshow(
        display_data.T,
        aspect='auto',
        origin='lower',
        cmap='inferno',
        extent=[0, cross.shape[0]*dz*1e3, -1, 1]
    )
    axs[1].set_xlabel("Propagation distance (mm)")
    axs[1].set_ylabel("x cross-section")
    axs[1].set_title("Beam Propagation Cross-section")
    plt.colorbar(im2, ax=axs[1], label="log10(Intensity)")
    
    # Output field (phase)
    im1 = axs[2].imshow(np.abs(U_final), cmap="twilight")
    axs[2].set_title("Output Phase")
    plt.colorbar(im1, ax=axs[2])

    plt.tight_layout()
    plt.show()
    
thin_lens_demo(U0, 10e-3)

## Discussion
TODO

# Part 2
## Simulating the Thick Lens

In this part we will move to the significantly more complicated task of simulating a thick lens. This will be done in a similar manner to the thin lens, but now we will need to define phase masks for each individual slice of the thick lens and propagate with the beam propagation method. Additionally, we must consider that the step size we will need to properly resolve the features of the lens will be incredibly small, and therefore prohibitively slow if we use it for the whole simulation over four focal lengths. To overcome this, we will use a different step size for the regions of free space propagation than in the region inside the lens. Take a moment to look at the function propagate_thick_lens_system to understand how this is done.

In this section you will implement the function generate_thick_lens_phase_slices. This function is responsible for generating the phase masks that encode the whole structure of the lens. These phase masks will span a z range of 0 to thickness, which is a passed parameter that encodes the maximum thickness of the lens. The propagation routine that is provided will automatically handle the thickness of the lens, so you will not need to write any explicit logic to handle this.

To implement the generate_thick_lens_phase_slices function, follow this logic:
1) Start by assuming that we have a plano-convex lens, i.e. one side is flat, and the other side is curved. For this problem we will assume that the left side of the lens is flat and the right side is curved. Additionally, we assume that the right surface has spherical curvature.
2) Calculate the radius of curvature of the right surface as a function of n_lens and the focal length f. To do this, recall the Lensmaker's equation 
$$
\frac{1}{f} = (n - 1)\left(\frac{1}{R_1} - \frac{1}{R_2} + \frac{(n - 1)d}{n R_1 R_2}\right)
$$
for left and right radii $R_1$ and $R_2$, respectively, refractive index n, and lens thickness d.
3) Define the thickness of the lens as a function of X and Y. This value can be calculated from the total thickness of the lens and the surface curvature $R_2$.
4) Generate each slice by checking for each coordinate at a given z value if z is less than the thickness of that location and assigning the correct refractive index.
5) Convert the slices into phase masks from their relative refractive indices and the simulation step size.

After you have implemented the function generate_thick_lens_phase_slices, you should use the provided plotting routine, which will show you a cross-section of the lens, to verify that what you have made looks reasonable. Feel free to play around with the parameters of the sample plot to make sure that you are happy with your implementation.

Once you are happy with your phase mask implementation, you can run the remaining provided functions to simulate the whole system and observe its output. Then, answer the following questions:
1) What do you immediately notice about the difference between the thin lens and the thick lens output? How would you describe the aberrations present? (look at https://en.wikipedia.org/wiki/Optical_aberration if you are not familiar with the common terminology)
2) Try changing the thickness of the lens. Does this seem to have a big effect on the output? What about the split-step method would make this the case? (Note that the principal planes of the lens are already factored in, so the propagation is guaranteed to be to the focal point)
3) If you were designing a real lens that you wanted to simulate with this code, what are some changes you could make to the design to make the simulation run as quickly as possible for a fixed focal length?


In [ ]:
def generate_thick_lens_phase_slices(z_steps, n_lens, f, thickness):
    """
    Generate BPM phase slices for a thick lens.
    """
    raise NotImplementedError

In [ ]:
# - No modification necessary - 

def plot_lens_cross_section(phase_slices, x_coords, dz):
    """
    Plot the x–z cross-section of a thick lens phase stack.
    
    Parameters
    ----------
    phase_slices : list or array
        List of 2D phase arrays (shape: Nx x Ny) along z
    x_coords : array
        Physical transverse x coordinates (meters)
    dz : float
        Step size along z (meters)
    """
    
    # Convert list -> 3D array: (z, x, y)
    phase = np.stack(phase_slices, axis=0)  # (z, x, y)
    
    # Take the central y slice
    y_mid = phase.shape[2] // 2
    xz = phase[:, :, y_mid]   # shape: (z, x)
    
    # Transpose so z is horizontal and x is vertical
    xz_plot = xz.T  # now shape (x, z)
    
    # Generate real z coordinates
    z_coords = np.arange(len(phase_slices)) * dz
    
    plt.figure(figsize=(6,5))
    im = plt.imshow(
        xz_plot,
        extent=[z_coords.min()*1e3, z_coords.max()*1e3, x_coords.min()*1e3, x_coords.max()*1e3],
        origin='lower',
        cmap='RdBu',
        aspect='equal'
    )
    plt.xlabel("Propagation distance z (mm)")
    plt.ylabel("Transverse position x (mm)")
    plt.title("Thick Lens Phase x–z Cross Section")
    plt.colorbar(im, label="Phase [rad]")
    plt.tight_layout()
    plt.show()
    

# - Modify these parameters - 
lens_steps = 400
lens_n = 1.1
lens_focal_length = 10e-3
lens_thickness = 0.5e-3
lens_phase_slices = generate_thick_lens_phase_slices(lens_steps, lens_n, lens_focal_length, lens_thickness)
dz = lens_thickness/lens_steps
plot_lens_cross_section(lens_phase_slices, x_coords=x, dz=dz)

In [ ]:
# - No modification necessary -

def propagate_thick_lens_system(U0, f=10e-3, free_space_steps=200, lens_steps=400, n_lens=1.1, lens_thickness=0.5e-3):

    U = U0.copy()
    phase_slices = generate_thick_lens_phase_slices(lens_steps, n_lens, f, lens_thickness)
    
    h1 = lens_thickness/lens_n
    h2 = 0

    # ---------- free space before lens ----------
    dz_1 = (2*f-h1) / free_space_steps
    z1 = dz_1 * np.arange(1, free_space_steps + 1)
    cross1, U = bpm_propagate(U, dz_1, steps=free_space_steps)
    
    # ---------- lens ----------
    dz_2 = lens_thickness / lens_steps
    z2 = z1[-1] + dz_2 * np.arange(1, lens_steps + 1)
    cross2, U = bpm_propagate(U, dz_2, phase_slices=phase_slices)
    
    # ---------- free space after lens ----------
    dz_3 = (2*f-h2) / free_space_steps
    z3 = z2[-1] + dz_3 * np.arange(1, free_space_steps + 1)
    cross3, U = bpm_propagate(U, dz_3, steps=free_space_steps)
    
    z_positions = np.concatenate([z1, z2, z3])
    cross_sections = np.vstack([cross1, cross2, cross3])
    
    return np.array(cross_sections), np.array(z_positions), U

In [ ]:
# - No modification necessary -

from scipy.interpolate import interp1d

def plot_thick_lens_results(U0, cross_sections, z_positions, U_final, x):

    # ---- create uniform z grid ----
    z_uniform = np.linspace(z_positions.min(), z_positions.max(), len(z_positions))

    interp = interp1d(z_positions, cross_sections, axis=0, kind='linear')
    cross_uniform = interp(z_uniform)

    # ---- log intensity for visualization ----
    display_data = np.log10(cross_uniform + 1e-12)

    # ---- plotting ----
    fig, axs = plt.subplots(1, 3, figsize=(18,5))

    # Input field (phase)
    im0 = axs[0].imshow(np.abs(U0), cmap="twilight")
    axs[0].set_title("Input Amplitude")
    plt.colorbar(im0, ax=axs[0])

    # Propagation cross-section
    im2 = axs[1].imshow(
        display_data.T,
        aspect='auto',
        origin='lower',
        cmap='inferno',
        extent=[
            z_uniform.min()*1e3,
            z_uniform.max()*1e3,
            x.min()*1e3,
            x.max()*1e3
        ]
    )

    axs[1].set_xlabel("Propagation distance (mm)")
    axs[1].set_ylabel("x position (mm)")
    axs[1].set_title("Beam Propagation Cross-section")

    plt.colorbar(im2, ax=axs[1], label="log10(Intensity)")

    # Output field (phase)
    im1 = axs[2].imshow(np.abs(U_final), cmap="twilight")
    axs[2].set_title("Output Amplitude")
    plt.colorbar(im1, ax=axs[2])

    plt.tight_layout()
    plt.show()


# - Modify these parameters - 

cross_sections, z_positions, U = propagate_thick_lens_system(U0, f=10e-3, n_lens=1.1, lens_thickness=0.5e-3)
plot_thick_lens_results(U0, cross_sections, z_positions, U, x)


## Discussion
TODO

# Bonus
In this simulation we have assumed that the refractive index of our lens is only 1.1. In reality, glass typically has a refractive index of around 1.5. If you try running the thick lens simulation with this refractive index, you will see that the output is significantly degraded. Can you expound on why you observe this and what you could change about the simulation to enable simulation with this high $\Delta n$?

## Discussion
TODO